In [1]:
import torch
import torch.nn as nn
import math

In [4]:
class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model=256, n_heads=8, n_groups=2):
        super().__init__()
        assert n_heads % n_groups == 0
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.n_groups = n_groups
        self.head_dim = d_model // n_heads
        self.heads_per_group = n_heads // n_groups
        
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, n_groups * self.head_dim)
        self.v_proj = nn.Linear(d_model, n_groups * self.head_dim)      
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, S, D = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, S, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, S, self.n_groups, self.head_dim).transpose(1, 2)
        v = v.view(B, S, self.n_groups, self.head_dim).transpose(1, 2)

        q = q.view(B, self.n_groups, self.heads_per_group, S, self.head_dim)
        k = k.unsqueeze(2)
        v = v.unsqueeze(2)

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn = torch.softmax(scores, dim=-1)
        out = torch.matmul(attn, v)
        out = out.view(B, self.n_heads, S, self.head_dim)
        out = out.transpose(1, 2).contiguous().view(B, S, D)
        
        return self.out_proj(out)